In [1]:
import dagshub
import mlflow

# Connect to your DagsHub repo
dagshub.init(repo_owner="shreychauhan02", repo_name="my_final_project", mlflow=True)



Accessing as shreychauhan02

Initialized MLflow to track repo "shreychauhan02/my_final_project"

Repository shreychauhan02/my_final_project initialized!

In [2]:
# Set or create an experiment
mlflow.set_experiment("Handling Imbalance Data")

2026/09/25 10:11:41 INFO mlflow.tracking.fluent: Experiment with name 'Handling Imbalance Data' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/af5cbdf3029b4072b9213ced222ffa97', creation_time=1790311301503, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1790311301503, lifecycle_stage='active', name='Handling Imbalance Data', tags={}, trace_location=None, workspace='default'>

In [3]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [4]:
df = pd.read_csv('../dataset/pre.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [5]:
def run_imbalanced_experiment(imbalance_method):
    ngram_range=(1,3)
    max_features=1000

    vectorizer=TfidfVectorizer(ngram_range=ngram_range,max_features=max_features)

    X_text=df['clean_comment']
    y=df['category']

    # Step 1: Split raw text FIRST, then fit vectorizer on train only
    X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.2, random_state=42, stratify=y)
    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 2: Handle imbalance on the TRAINING SET ONLY (never resample the test set)
    if imbalance_method=='class_weights':
        class_weight='balanced'
    else:
        class_weight=None
        if imbalance_method=='oversampling':
            smote=SMOTE(random_state=42)
            X_train, y_train = smote.fit_resample(X_train, y_train)
        elif imbalance_method=='adasyn':
            adasyn=ADASYN(random_state=42)
            X_train, y_train = adasyn.fit_resample(X_train, y_train)
        elif imbalance_method=='undersampling':
            rus = RandomUnderSampler(random_state=42)
            X_train, y_train = rus.fit_resample(X_train, y_train)
        elif imbalance_method=='smote_enn':
            smote_enn=SMOTEENN(random_state=42)
            X_train, y_train = smote_enn.fit_resample(X_train, y_train)

    # Step 3: Train and log with MLflow
    with mlflow.start_run() as run:
        mlflow.set_tag("mlflow.runName", f"Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "imbalance_handling")
        mlflow.set_tag("model_type", "RandomForestClassifier")
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, imbalance handling method={imbalance_method}")

        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("imbalance_method", imbalance_method)

        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, class_weight=class_weight)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, Imbalance={imbalance_method}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        mlflow.sklearn.log_model(model, f"random_forest_model_tfidf_trigrams_imbalance_{imbalance_method}")

# Step 4: Run experiments for different imbalance methods
imbalance_methods = ['class_weights', 'oversampling', 'adasyn', 'undersampling', 'smote_enn']

for method in imbalance_methods:
    run_imbalanced_experiment(method)

2026/09/25 10:13:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_class_weights_RandomForest_TFIDF_Trigrams at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/3/runs/e468cabb7f31473bab21040e52b0e9b4
🧪 View experiment at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/3


2026/09/25 10:17:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_oversampling_RandomForest_TFIDF_Trigrams at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/3/runs/4728fb319dbf4f20bc9574909839dba0
🧪 View experiment at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/3


2026/09/25 10:20:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_adasyn_RandomForest_TFIDF_Trigrams at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/3/runs/cbd0f4a8203d4b339096c98625768dd5
🧪 View experiment at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/3


2026/09/25 10:23:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_undersampling_RandomForest_TFIDF_Trigrams at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/3/runs/e0b0fd40f5ea4881a9a0445552028e43
🧪 View experiment at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/3


2026/09/25 10:26:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Imbalance_smote_enn_RandomForest_TFIDF_Trigrams at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/3/runs/ab78d9ede0a2419194bd624be4f5611b
🧪 View experiment at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/3
